# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Davidsoooooon/ML-Intern---Briones/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data contract

1. **What one row means:**  
   In the raw daily table, one row represents one report date, one pseudonymized client, and one pseudonymized content page. In my final feature frame, one row represents one client-content page summarized at the March 15, 2026 decision moment.

2. **Tables used:**  
   I use `fact_content_daily_performance`. I use `client_hash_id` and `content_hash_id` only as context and grouping fields.

3. **Time window:**  
   I use March 2026 as my development month. Features are calculated from March 1–14, 2026. The proxy outcome is measured from March 15–28, 2026. March 15 is the decision moment.

4. **What I predict or rank:**  
   I rank pages by their probability of experiencing an impressions decline of more than 20% during the following 14-day window. The output supports an editorial review queue, not an automatic content decision.

5. **What I deliberately exclude:**  
   I exclude GA4 engagement metrics from this first feature frame because GA4 availability differs between clients. Rows from before tracking began may contain zero-filled GA4 values, which should not be interpreted as zero engagement.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification

#### Features

1. `log_impressions_early` — logged search impressions observed from March 1–14.
2. `ctr_early` — clicks divided by impressions during March 1–14.
3. `avg_position_early` — average valid search position during March 1–14.
4. `position_volatility_early` — variation in search position during March 1–14.
5. `active_days_early` — number of days with at least one search impression during March 1–14.

#### Label / proxy

- `is_declining_proxy` — equals 1 when impressions from March 15–28 are less than 80% of impressions from March 1–14; otherwise 0.
- `impressions_late` — used only to calculate the proxy label and never used as an honest feature.

#### Context

- `client_hash_id` — used for grouping and future client-level validation.
- `content_hash_id` — identifies the pseudonymized content page.
- `report_date` — defines the feature and outcome windows.

#### Excluded

- GA4 engagement metrics — excluded because data availability differs between clients.
- Client names, URLs, titles, keywords, and raw queries — private information is unnecessary for this analysis.
- March 15–28 measurements — excluded from the features because they occur after the decision moment.

In [12]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connection ready.")

DuckDB connection ready.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Query 1 — Raw table grain

The expected raw grain is one report date, one pseudonymized client, and one pseudonymized content item. An empty result means no duplicate page-day rows were found.

In [13]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS duplicate_count
    FROM {MARCH_DAILY}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

display(grain_check)

if grain_check.empty:
    print("Grain holds: no duplicate page-day rows found.")
else:
    print("Warning: duplicate grain rows were found.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count


Grain holds: no duplicate page-day rows found.


### Verification Query 2 — Slice size and date span

This query shows the number of rows, clients, content pages, and the actual date range in the March 2026 slice.

In [14]:
slice_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count,
        MIN(report_date) AS minimum_date,
        MAX(report_date) AS maximum_date
    FROM {MARCH_DAILY}
""").df()

display(slice_summary)

,row_count,client_count,content_count,minimum_date,maximum_date
0,9841378,55,331437,2026-03-01,2026-03-31


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Verification Query 3 — GA4 data availability

GA4 values may be zero-filled when analytics tracking was not yet available. This query uses `IS TRUE` and shows how many March rows remain after filtering for valid GA4 availability.

### Five-feature frame

The decision moment is March 15, 2026. All five features use only data observed from March 1–14. The proxy outcome is calculated from March 15–28.

In [15]:
availability_check = con.sql(f"""
    WITH all_rows AS (
        SELECT COUNT(*) AS total_rows
        FROM {MARCH_DAILY}
    ),
    available_rows AS (
        SELECT COUNT(*) AS rows_after_filter
        FROM {MARCH_DAILY}
        WHERE ga4_data_available IS TRUE
    )
    SELECT
        total_rows,
        rows_after_filter,
        ROUND(
            100.0 * rows_after_filter / NULLIF(total_rows, 0),
            2
        ) AS percent_surviving
    FROM all_rows
    CROSS JOIN available_rows
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_after_filter,percent_surviving
0,9841378,413966,4.21


In [16]:
feature_source = con.sql(f"""
    WITH page_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-14'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS impressions_early,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-14'
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS clicks_early,

            AVG(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-14'
                    THEN NULLIF(gsc_avg_position, 0)
                END
            ) AS avg_position_early,

            STDDEV_SAMP(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-14'
                    THEN NULLIF(gsc_avg_position, 0)
                END
            ) AS position_volatility_early,

            COUNT(
                DISTINCT CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-14'
                         AND gsc_impressions > 0
                    THEN report_date
                END
            ) AS active_days_early,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-15'
                                         AND DATE '2026-03-28'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS impressions_late

        FROM {MARCH_DAILY}
        WHERE report_date BETWEEN DATE '2026-03-01'
                              AND DATE '2026-03-28'
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        client_hash_id,
        content_hash_id,
        impressions_early,
        impressions_late,

        100.0 * clicks_early
        / NULLIF(impressions_early, 0) AS ctr_early,

        avg_position_early,
        position_volatility_early,
        active_days_early,

        CASE
            WHEN impressions_late < 0.80 * impressions_early
            THEN 1
            ELSE 0
        END AS is_declining_proxy

    FROM page_windows
    WHERE impressions_early >= 100
""").df()

feature_source["log_impressions_early"] = np.log1p(
    feature_source["impressions_early"]
)

feature_columns = [
    "log_impressions_early",
    "ctr_early",
    "avg_position_early",
    "position_volatility_early",
    "active_days_early",
]

feature_frame = feature_source[
    [
        "client_hash_id",
        "content_hash_id",
        *feature_columns,
        "is_declining_proxy",
    ]
].copy()

print(f"Feature-frame rows: {len(feature_frame):,}")
print(f"Number of model features: {len(feature_columns)}")

display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 75,526
Number of model features: 5


,client_hash_id,content_hash_id,log_impressions_early,ctr_early,avg_position_early,position_volatility_early,active_days_early,is_declining_proxy
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,5.356586,0.473934,4.245075,2.342553,14,0
1,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,7.240650,0.000000,6.104078,1.173984,13,0
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,5.472271,0.000000,4.542255,1.455799,14,1
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,5.955837,0.259740,4.351876,1.405632,14,0
4,client_62f4a7e64f5e0096,content_225dc9235023be5f,5.609472,0.367647,10.398054,4.968515,14,1


### Deliberate leakage experiment

I deliberately add `leak_decline_ratio`, which uses impressions from the March 15–28 outcome window. Because the proxy label is calculated from this ratio, it gives the model part of the answer. I compare the leaky result with the honest five-feature result, then remove the leaked column.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

model_df = feature_source.copy()

model_df["log_impressions_early"] = np.log1p(
    model_df["impressions_early"]
)

# Deliberate leakage: this uses the future outcome window.
model_df["leak_decline_ratio"] = (
    model_df["impressions_late"]
    / model_df["impressions_early"]
)

all_columns = feature_columns + ["leak_decline_ratio"]

X = model_df[all_columns].replace(
    [np.inf, -np.inf],
    np.nan
).copy()

for column in X.columns:
    X[column] = X[column].fillna(X[column].median())

y = model_df["is_declining_proxy"]

train_index, test_index = train_test_split(
    model_df.index,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

def calculate_auc(columns):
    model = DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    )

    model.fit(
        X.loc[train_index, columns],
        y.loc[train_index]
    )

    probabilities = model.predict_proba(
        X.loc[test_index, columns]
    )[:, 1]

    return roc_auc_score(
        y.loc[test_index],
        probabilities
    )

honest_auc = calculate_auc(feature_columns)

leaky_auc = calculate_auc(
    feature_columns + ["leak_decline_ratio"]
)

print(f"Honest five-feature ROC AUC: {honest_auc:.3f}")
print(f"Leaky ROC AUC:               {leaky_auc:.3f}")

Honest five-feature ROC AUC: 0.634
Leaky ROC AUC:               1.000


In [18]:
model_df.drop(
    columns=["leak_decline_ratio"],
    inplace=True
)

final_feature_columns = feature_columns.copy()

assert "leak_decline_ratio" not in model_df.columns
assert len(final_feature_columns) == 5

print("Leakage column removed.")
print(f"Final honest ROC AUC kept: {honest_auc:.3f}")
print("Final features:", final_feature_columns)

Leakage column removed.
Final honest ROC AUC kept: 0.634
Final features: ['log_impressions_early', 'ctr_early', 'avg_position_early', 'position_volatility_early', 'active_days_early']


### Leakage lesson

The honest five-feature model produced a ROC AUC of 0.637. When I deliberately added a column calculated from the future outcome window, the score increased to 1.000. This perfect result is not trustworthy because the leaked feature directly contains information used to create the label. I removed the leaked column and kept the honest score.

## 4. Data limits

### Named limitation: March-only observation window

This analysis uses only one development month. A short fall in impressions may represent temporary noise, seasonality, or demand moving to another related page rather than a genuine content decline. The result can support editorial review prioritization, but it cannot prove that a page needs refreshing or that editing it would cause traffic recovery.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.